# Daily Challenge: Preprocess & Fine-tune Transformer-based Models (BERT and XLM-RoBERTa)

Week 12, Day 4

**Dataset:** multilingual **Natural Language Inference** ("Contradictory, My Dear Watson"). Each example is a pair of sentences, a **premise** and a **hypothesis**, in one of 15 languages, and the label says how they are related:
- **0 = entailment**: the hypothesis follows from the premise;
- **1 = neutral**: the hypothesis may or may not be true;
- **2 = contradiction**: the hypothesis contradicts the premise.

Plan: (1) understand BERT and XLM-RoBERTa, (2) tokenize text, (3) prepare model inputs, (4) load and explore the dataset, (5) create 5 stratified cross-validation folds, then (6) fine-tune a multilingual transformer on one fold and (7) predict the test set.

In [ ]:
%pip install --quiet transformers sentencepiece

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Understanding BERT and XLM-RoBERTa

**BERT** (Bidirectional Encoder Representations from Transformers, Google 2018) is an **encoder-only Transformer**. Each token attends to all the other tokens on its left and right (bidirectional self-attention), so every word gets a representation that depends on its whole context. It is pre-trained on English Wikipedia and BooksCorpus with two objectives: **Masked Language Modeling** (predict 15% of masked tokens) and **Next Sentence Prediction**. It uses a **WordPiece** tokenizer (~30,000 tokens) where word continuations start with `##`, and special tokens `[CLS]` (start, used for classification), `[SEP]` (separator / end), `[PAD]`, `[MASK]`, `[UNK]`. Because it was trained on sentence pairs, its tokenizer also produces `token_type_ids` (segment A = 0, segment B = 1).

**XLM-RoBERTa** (Facebook AI 2019) uses the same encoder architecture but is trained the **RoBERTa** way (more data, longer, dynamic masking, MLM only, no NSP) on **2.5 TB of CommonCrawl text in 100 languages**. It uses a **SentencePiece** tokenizer with a shared vocabulary of **250,002 tokens** for all languages (word starts are marked with `▁`), and RoBERTa-style special tokens `<s>` (start), `</s>` (end / separator), `<pad>`, `<mask>`, `<unk>`. It does not use `token_type_ids`. Thanks to its multilingual pre-training, a model fine-tuned in one language transfers well to the others (cross-lingual transfer), which is ideal for our dataset.

**Pre-trained versions:**

| Checkpoint | Layers / hidden size | Parameters | Characteristics |
|---|---|---|---|
| `bert-base-uncased` / `bert-base-cased` | 12 / 768 | 110M | English only; *uncased* lower-cases the text, *cased* keeps capital letters (useful for names) |
| `bert-large-uncased` | 24 / 1024 | 340M | English, more accurate but slower |
| `bert-base-multilingual-cased` (mBERT) | 12 / 768 | 178M | 104 languages, trained on Wikipedia |
| `distilbert-base-multilingual-cased` | 6 / 768 | 135M | Distilled mBERT: ~2× faster, slightly less accurate |
| `xlm-roberta-base` | 12 / 768 | 278M | 100 languages, CommonCrawl, much stronger than mBERT on non-English text |
| `xlm-roberta-large` | 24 / 1024 | 560M | Best accuracy for multilingual tasks (state of the art on XNLI), needs a GPU |

In [ ]:
from transformers import BertTokenizer, XLMRobertaTokenizer

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

for name, tok in [("BERT (bert-base-uncased)", bert_tokenizer), ("XLM-RoBERTa (xlm-roberta-base)", xlmr_tokenizer)]:
    print(f"{name}: vocab_size = {tok.vocab_size:,}")
    print("  special tokens:", tok.special_tokens_map)

## 2. Tokenizing Text

In [ ]:
sentence = "Transformers are revolutionizing natural language processing!"

for name, tok in [("BERT", bert_tokenizer), ("XLM-R", xlmr_tokenizer)]:
    enc = tok.encode_plus(sentence)
    print(f"--- {name} ---")
    print("tokens        :", tok.convert_ids_to_tokens(enc["input_ids"]))
    print("input_ids     :", enc["input_ids"])
    print("attention_mask:", enc["attention_mask"])
    if "token_type_ids" in enc:
        print("token_type_ids:", enc["token_type_ids"])
    print("decoded       :", tok.decode(enc["input_ids"]))

- **`input_ids`**: the index of each token in the vocabulary, including the special tokens added automatically: BERT wraps the sentence as `[CLS] … [SEP]` (ids 101 and 102), XLM-R as `<s> … </s>` (ids 0 and 2).
- **`attention_mask`**: 1 for real tokens, 0 for padding (no padding here, so only 1s).
- **`token_type_ids`** (BERT only): which segment each token belongs to.
- **`labels`** are not produced by the tokenizer: they are the target classes we add ourselves when building the training dataset (here the NLI label 0/1/2).
- BERT uncased lower-cases the text and splits rare words into WordPieces (`revolution`, `##izing`), XLM-R keeps the case and marks the beginning of words with `▁`.

In [ ]:
# Two-sentence (pair) tokenization: exactly what NLI needs (premise + hypothesis)
premise = "The cat is sleeping on the sofa."
hypothesis = "An animal is resting."

for name, tok in [("BERT", bert_tokenizer), ("XLM-R", xlmr_tokenizer)]:
    enc = tok.encode_plus(premise, hypothesis)
    print(f"--- {name} ---")
    print("tokens        :", tok.convert_ids_to_tokens(enc["input_ids"]))
    if "token_type_ids" in enc:
        print("token_type_ids:", enc["token_type_ids"])
    print("decoded       :", tok.decode(enc["input_ids"]))

# The same multilingual pair in French: XLM-R handles every language with the same vocabulary
fr = xlmr_tokenizer.encode_plus("Le chat dort sur le canapé.", "Un animal se repose.")
print("\nXLM-R (French):", xlmr_tokenizer.convert_ids_to_tokens(fr["input_ids"]))
print("BERT uncased (French):", bert_tokenizer.tokenize("Le chat dort sur le canapé."))

For a **pair of sentences**, BERT builds `[CLS] premise [SEP] hypothesis [SEP]` and marks the hypothesis tokens with `token_type_ids = 1`; XLM-R builds `<s> premise </s></s> hypothesis </s>` (a double separator, no segment ids). The model therefore sees both sentences at once and its self-attention can compare every word of the hypothesis with every word of the premise, which is how it learns entailment and contradiction. The English BERT tokenizer splits French words into many pieces and loses accents, while XLM-R tokenizes French naturally.

## 3. Preparing Input Data for the Model

In [ ]:
MAX_LEN = 32
for name, tok in [("BERT", bert_tokenizer), ("XLM-R", xlmr_tokenizer)]:
    enc = tok.encode_plus(
        premise, hypothesis,
        add_special_tokens=True,     # [CLS]/[SEP] or <s>/</s>
        max_length=MAX_LEN,
        padding="max_length",        # pad short inputs up to MAX_LEN
        truncation=True,             # cut long inputs down to MAX_LEN
        return_attention_mask=True,
        return_tensors="pt",
    )
    print(f"--- {name} (max_length={MAX_LEN}) ---")
    print("input_ids     :", enc["input_ids"][0].tolist())
    print("attention_mask:", enc["attention_mask"][0].tolist())
    print("padding token :", tok.pad_token, "=", tok.pad_token_id, "| real tokens:", int(enc["attention_mask"].sum()))

# Truncation of a long text
long_text = "This sentence is far too long for the model. " * 20
enc = xlmr_tokenizer.encode_plus(long_text, max_length=MAX_LEN, truncation=True, padding="max_length")
print("\nTruncated text:", xlmr_tokenizer.decode(enc["input_ids"]), "| length:", len(enc["input_ids"]))

print("\nXLM-R special tokens:", xlmr_tokenizer.special_tokens_map)
print("bos <s> =", xlmr_tokenizer.bos_token_id, "| eos </s> =", xlmr_tokenizer.eos_token_id,
      "| pad =", xlmr_tokenizer.pad_token_id, "| vocab_size =", xlmr_tokenizer.vocab_size)

- **Padding and truncation:** a batch must be a rectangular tensor, so every sequence is brought to the same length: shorter ones are **padded** with the pad token (`[PAD]` = 0 for BERT, `<pad>` = 1 for XLM-R) and longer ones are **truncated** (the end of the text is cut, keeping the final `</s>`).
- **Special tokens:** `<s>` marks the beginning of the sequence; its final representation is used as the summary of the whole input for classification (like `[CLS]` in BERT). `</s>` marks the end of each segment.
- **`attention_mask`:** it tells the model which positions are real (1) and which are padding (0). Inside self-attention, the padding positions get a score of −∞ before the softmax, so they receive zero weight: the padding does not change the representation of the real tokens and does not influence the prediction.

## 4. Loading and Exploring the Dataset

In [ ]:
train = pd.read_csv("data/train.csv.zip")
test = pd.read_csv("data/test.csv.zip")
print("train:", train.shape, "| test:", test.shape)
display(train.head())
display(test.head())
print("Missing values:", train.isna().sum().sum(), "/", test.isna().sum().sum())

In [ ]:
label_names = {0: "entailment", 1: "neutral", 2: "contradiction"}
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
train['label'].map(label_names).value_counts().plot(kind='bar', ax=axes[0], rot=0, color='tab:blue', title='Label distribution (train)')
train['language'].value_counts().plot(kind='bar', ax=axes[1], color='tab:green', title='Languages (train)')
pair_len = [len(xlmr_tokenizer.encode(p, h)) for p, h in zip(train['premise'], train['hypothesis'])]
axes[2].hist(pair_len, bins=60, color='tab:orange'); axes[2].set_title('XLM-R tokens per (premise, hypothesis) pair')
plt.tight_layout(); plt.show()

print(train['label'].map(label_names).value_counts(normalize=True).round(3))
print(f"Pair length (XLM-R tokens): median {int(np.median(pair_len))}, 95th pct {int(np.percentile(pair_len, 95))}, "
      f"99th pct {int(np.percentile(pair_len, 99))}, max {max(pair_len)}")
for lang in ['English', 'French', 'Arabic']:
    r = train[train['language'] == lang].iloc[0]
    print(f"\n[{lang}] premise: {r['premise']}\n          hypothesis: {r['hypothesis']}\n          label: {label_names[r['label']]}")

- **Columns:** `id`, `premise`, `hypothesis`, `lang_abv`, `language` and (train only) `label`. For training we need the two **text columns `premise` and `hypothesis`**, encoded together as a pair, and the **`label`** column; `language` is useful to stratify and analyse, `id` to build the submission.
- **12,120 training pairs** and **5,195 test pairs** (without labels), no missing values.
- The three labels are **balanced** (34% entailment, 32% neutral, 34% contradiction), so accuracy is a good metric.
- **15 languages**, but English represents 57% of the data; the other 14 languages have about 340–410 pairs each. A multilingual model (mBERT, XLM-R) is required.
- Pairs are short: almost all fit in **128 tokens**, so this is a good `max_length`.

## 5. Creating Cross-Validation Folds

In [ ]:
from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_folds, valid_folds = [], []
for fold, (train_idx, valid_idx) in enumerate(kf.split(train, train['label'])):
    train_folds.append(train.iloc[train_idx].reset_index(drop=True))
    valid_folds.append(train.iloc[valid_idx].reset_index(drop=True))

rows = []
for i, (tr, va) in enumerate(zip(train_folds, valid_folds)):
    dist = va['label'].value_counts(normalize=True).sort_index()
    rows.append({'fold': i + 1, 'train size': len(tr), 'valid size': len(va),
                 'valid % entailment': dist[0], 'valid % neutral': dist[1], 'valid % contradiction': dist[2],
                 'valid % English': (va['language'] == 'English').mean()})
folds_df = pd.DataFrame(rows).set_index('fold')
display(folds_df.style.format({c: '{:.1%}' for c in folds_df.columns if '%' in c}))
print("Overall label distribution:", train['label'].value_counts(normalize=True).sort_index().round(3).to_dict())
assert sum(len(v) for v in valid_folds) == len(train)  # every example is used exactly once for validation

`StratifiedKFold(n_splits=5, shuffle=True)` splits the 12,120 pairs into 5 folds of about 2,424 validation / 9,696 training examples. Stratification keeps the **same label distribution** (≈34 / 32 / 34%) in every fold, and every example appears in exactly one validation set. Training one model per fold and averaging their scores (or their predictions on the test set) gives a more reliable estimate than a single split.

## 6. Fine-tuning a Multilingual Transformer (fold 1)

To keep this notebook runnable on a CPU, we fine-tune **`distilbert-base-multilingual-cased`** (6 layers) on **fold 1 only**, for 2 epochs. On a Colab **GPU** you can set `MODEL_NAME = "xlm-roberta-base"` (or `"xlm-roberta-large"`), more epochs and loop over the 5 folds: XLM-R is clearly more accurate on multilingual NLI.

We use **dynamic padding**: each batch is padded only to the length of its longest pair (max 128), which is much faster than padding everything to 128.

In [ ]:
import time
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup

MODEL_NAME = "distilbert-base-multilingual-cased"   # on a GPU: "xlm-roberta-base"
MAX_LEN, BATCH_SIZE, EPOCHS, LR, FOLD = 128, 32, 2, 3e-5, 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NLIDataset(Dataset):
    def __init__(self, df, with_labels=True):
        self.enc = tokenizer(list(df['premise']), list(df['hypothesis']), truncation=True, max_length=MAX_LEN)
        self.labels = df['label'].tolist() if with_labels else None
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = self.labels[i]
        return item

collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(NLIDataset(train_folds[FOLD]), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
valid_loader = DataLoader(NLIDataset(valid_folds[FOLD]), batch_size=64, collate_fn=collator)

torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
print(f"{MODEL_NAME}: {model.num_parameters():,} parameters | {len(train_loader)} steps per epoch | device: {device}")

In [ ]:
from sklearn.metrics import accuracy_score

def predict(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            y = batch.pop('labels', None)
            logits = model(**{k: v.to(device) for k, v in batch.items()}).logits
            preds.append(logits.argmax(-1).cpu().numpy())
            if y is not None:
                labels.append(y.numpy())
    return np.concatenate(preds), (np.concatenate(labels) if labels else None)

history = []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        running += loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch} step {step}/{len(train_loader)} | loss {running / step:.4f} | {(time.time() - t0) / 60:.1f} min")
    val_pred, val_true = predict(model, valid_loader)
    acc = accuracy_score(val_true, val_pred)
    history.append({'epoch': epoch, 'train loss': running / len(train_loader), 'valid accuracy': acc})
    print(f"==> epoch {epoch}: train loss {running / len(train_loader):.4f} | validation accuracy {acc:.4f}")
print(f"Training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(val_true, val_pred, target_names=list(label_names.values()), digits=4))
ConfusionMatrixDisplay(confusion_matrix(val_true, val_pred), display_labels=list(label_names.values())).plot(cmap='Blues', colorbar=False)
plt.title(f'Fold {FOLD + 1} validation confusion matrix'); plt.show()

va = valid_folds[FOLD].assign(pred=val_pred)
by_lang = va.groupby('language').apply(lambda g: pd.Series({'n': len(g), 'accuracy': (g['pred'] == g['label']).mean()})).sort_values('accuracy')
display(by_lang.style.format({'n': '{:.0f}', 'accuracy': '{:.1%}'}))

## 7. Predictions on the Test Set

In [ ]:
test_loader = DataLoader(NLIDataset(test, with_labels=False), batch_size=64, collate_fn=collator)
test_pred, _ = predict(model, test_loader)
submission = pd.DataFrame({'id': test['id'], 'prediction': test_pred})
submission.to_csv('submission.csv', index=False)
display(submission.head())
print(submission['prediction'].map(label_names).value_counts(normalize=True).round(3))

os.makedirs('artifacts', exist_ok=True)
model.save_pretrained('artifacts/nli_model'); tokenizer.save_pretrained('artifacts/nli_model')
print("Model saved to artifacts/nli_model")

CONCLUSION_PLACEHOLDER